## Load the domain, generate and verify tasks.

This notebook walks through the **benchmark creation** path on `domains/mock_credit_union`:

1. Load and validate the domain bundle (policy, DB, tools, task types, seed tasks)
2. Inspect a seed task’s oracle actions and communicate criteria
3. Replay tools in the Environment and compute a target DB hash
4. Sample constraints and generate synthetic task drafts via the LLM
5. Run the verification gate (policy rules → task-type write rules → replay) and write passing tasks to `data/benchmarks/mock_credit_union/tasks.json`
6. Optionally reload and re-verify saved tasks

In [ ]:
import json
import pprint
from pathlib import Path

from aieng.syn_data.synbench.path_utils import find_repo_root, use_repo_root


# Set the root directory for this implementation
use_repo_root(Path("."))
ROOT = find_repo_root() / "implementations" / "agent_benchmark_generation"


DOMAIN_PATH = ROOT / "domains" / "mock_credit_union"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_credit_union"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

---
## Step 1 — Load and inspect the domain

A **domain bundle** is the simulated business. `load_domain()` reads all required files and returns a `DomainBundle` object used everywhere else.

In [ ]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

print("Domain name:", domain.manifest.name)
print("\nTools:")
for t in domain.tools:
    print(f"  - {t.name} ({t.tool_type.value})")
print("\nTask types:", list(domain.task_types.keys()))
print("\nSeed task IDs:", [t.id for t in domain.seed_tasks])
print("\nCustomers in DB:", list(domain.db["customers"].keys()))

In [ ]:
# Peek at policy and one customer
print("=== Policy (first 400 chars) ===")
print(domain.policy[:400], "...\n")

print("=== Sample customer cu_44026 ===")
print(json.dumps(domain.db["customers"]["cu_44026"], indent=2))

---
## Step 2 — Understand a Task and its oracle

Tasks are defined in `tasks.seed.json`. The **`evaluation_criteria.actions`** field is the **oracle** — the reference tool trace used for verification and scoring.

`user_scenario` keeps identity and style separate:
- **`user_name`** — customer identity (e.g. Alice Chen)
- **`personality_style`** — interaction style key from `user_simulator.yaml` (e.g. rushed)
- **`instructions`** — full clean goal with ids/args (simulator / oracle grounding; not shown to the agent)
- **`initial_message`** — styled first utterance; prefer incomplete (withhold ids/args for multi-turn)


In [ ]:
seed_task = next(t for t in domain.seed_tasks if t.id == "seed_fee_waiver")

print("Task ID:", seed_task.id)
print("Description:", seed_task.description)
print("Task type:", seed_task.task_type)
# Name and style are separate fields on the task
print("User name:", seed_task.user_scenario.user_name)
print("Personality style:", seed_task.user_scenario.personality_style)
print("Instructions (clean goal):", seed_task.user_scenario.instructions)
print("Initial message (styled):", seed_task.user_scenario.initial_message)
print("\nOracle actions:")
for i, a in enumerate(seed_task.evaluation_criteria.actions, 1):
    print(f"  {i}. {a.name}({a.arguments})")
print("\nMust communicate:", seed_task.evaluation_criteria.communicate_info)
print("Reward basis:", [r.value for r in seed_task.evaluation_criteria.reward_basis])

---
## Step 3 — Environment: replay tools and hash the database

The **Environment** clones `db.json` and dispatches tool calls via the domain's `ToolKit`. **Replay** runs the oracle on a fresh DB; **db_hash** fingerprints the final state for scoring.

In [ ]:
from aieng.syn_data.synbench.environment.core import Environment, replay_actions
from aieng.syn_data.synbench.environment.hashing import db_hash


oracle_env = replay_actions(domain, seed_task.evaluation_criteria.actions)
target_hash = db_hash(oracle_env.db)

print("Fee waived after oracle replay:", oracle_env.db["fees"]["fee_44026_1"]["waived"])
print("Balance after oracle replay:", oracle_env.db["accounts"]["acct_44026"]["current_balance"])
print("Target DB hash:", target_hash[:32], "...")

In [ ]:
# Wrong action: waive a late fee for a customer who already had one waived within
# 12 months (should be rejected by the tool's policy guard)
from aieng.syn_data.synbench.schemas.actions import Action


bad_env = Environment(domain)
try:
    bad_env.dispatch(
        Action(
            name="waive_fee",
            arguments={"customer_id": "cu_52371", "fee_id": "fee_52371_1", "reason": "first_time_request"},
        )
    )
except Exception as e:
    print("Expected error:", e)


---
## Step 4 — Generate synthetic tasks

**Generation** asks an LLM for a new `Task`: user scenario + oracle actions.

`ConstraintSampler` picks:
1. `task_type` 
2. a record from the primary collection (`case_index` for the credit union domain) + the customer ID related to that record (`entities`)
3. a `personality_style` from `user_simulator.yaml` → `personality_styles`

The generator fills `user_name` (from the related user row) and `personality_style` separately, keeps full ids/args in `instructions`, and styles an incomplete `initial_message` so the agent must elicit details.


In [41]:
from aieng.syn_data.synbench.generation.sampler import ConstraintSampler


# Sample task_type, entity IDs, and personality_style together
sampler = ConstraintSampler(domain, seed=42)
constraints = sampler.sample()
print("Sampled constraints:")
pprint.pprint(vars(constraints), sort_dicts=False)
print("\nPersonality style name:", (constraints.personality_style or {}).get("name"))

Sampled constraints:
{'task_type': 'inquiry',
 'entities': {'order_id': 'ord_1001', 'user_id': 'user_alice'},
 'allow_write': False,
 'primary_id': 'ord_1001',
 'entity_context': {'order_id': 'ord_1001',
                    'user_id': 'user_alice',
                    'status': 'pending',
                    'shipping_address': '123 Main St, Boston, MA',
                    'users': {'name': 'Alice Chen',
                              'email': 'alice@example.com'}},
 'personality_style': {'name': 'rule_breaker',
                       'description': 'Tries to bend or break policy; requests '
                                      'exceptions, refunds, or cancellations '
                                      'that may not be allowed, and argues '
                                      'when refused.'}}

Personality style name: rule_breaker


### Generate one task given the sampled constraints

In [42]:
from aieng.syn_data.synbench.generation.generator import TrajectoryGenerator


# Generate one draft
gen = TrajectoryGenerator(domain, seed=42)
draft = gen.generate_one("notebook_draft_001", constraints)
pprint.pprint(draft.model_dump(mode="json"), sort_dicts=False)

{'id': 'notebook_draft_001',
 'description': 'Inquire about the status of an existing order',
 'task_type': 'inquiry',
 'user_scenario': {'user_name': 'Alice Chen',
                   'personality_style': 'rule_breaker',
                   'instructions': 'Inquire about the status of order '
                                   'ord_1001. If the agent asks for '
                                   'identification, provide the name Alice '
                                   'Chen. If the agent asks for the order ID, '
                                   'provide ord_1001. If the agent refuses any '
                                   'request or tries to guide you through '
                                   'formal verification steps, complain that '
                                   'you are a VIP customer and demand '
                                   'immediate answers without the formality.',
                   'initial_message': "I need to know what's going on with my "
             

In [8]:
from aieng.syn_data.synbench.generation.generator import GenerationRun


# Generate N tasks
run = GenerationRun(domain, seed=42)
# run.run_and_verify(n=3) would generate and verify in one step instead.
drafts = run.run(n=3)
# Print all of the drafts
for i, draft in enumerate(drafts):
    print(f"Draft number {i + 1}:")
    pprint.pprint(draft.model_dump(mode="json"), sort_dicts=False)
    print("\n")

Draft number 1:
{'id': 'gen_1c6aa539_0',
 'description': 'Inquire about the status of a specific order',
 'task_type': 'inquiry',
 'user_scenario': {'user_name': 'Alice Chen',
                   'personality_style': 'rule_breaker',
                   'instructions': 'Inquire about the status of order '
                                   'ord_1001. If the agent asks for '
                                   "identification, provide the name 'Alice "
                                   "Chen'. If the agent is slow, push them to "
                                   'prioritize my request immediately.',
                   'initial_message': 'I demand to know the status of my order '
                                      'right now! Stop wasting my time and '
                                      'tell me where my package is.'},
 'evaluation_criteria': {'actions': [{'name': 'find_user_id',
                                      'arguments': {'name': 'Alice Chen'}},
                            

---
## Step 5 — Verification pipeline (quality gate) overview

Before a draft becomes a benchmark task, three checks run in `verify_draft()`:

1. Domain-specific **policy rules** if any exist — e.g. in the credit union domain, `verify_identity` must come first and match the DB. These rules live in `verify.py`.
2. **task_type validation** — `allow_write` in `task_types.yaml`: read-only types must not use WRITE tools; write types need at least one.
3. **Replay** — oracle actions must execute without error; compute `target_db_hash`

In [43]:
# Verify the generated tasks, and save them in OUT_DIR/tasks.json
verified_tasks, rejected_drafts = run.verify_drafts(drafts)
print(f" ({len(verified_tasks)} / 3) passed the verification pipeline.")
tasks_path = run.write_tasks(verified_tasks, OUT_DIR)
print(f"Written to: {tasks_path}")
print("Task IDs:", [t.id for t in verified_tasks])

 (3 / 3) passed the verification pipeline.
Written to: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_retail/tasks.json
Task IDs: ['gen_739597cc_0', 'gen_dae9dfa7_1', 'gen_d5b1dd1e_2']


In [44]:
# Inspect the rejected tasks if any
if rejected_drafts:
    print("\nRejected tasks:")
    for i, (draft, reason) in enumerate(rejected_drafts, 1):
        print(f"  {i}. {draft.id} ({reason})")
        print(f"    User scenario: {draft.user_scenario.initial_message}")
        print(f"    Oracle actions: {draft.evaluation_criteria.actions}")
else:
    print("No rejected tasks.")

No rejected tasks.


# Step 6 - Load previously generated tasks, re-run verification
Before running evaluation, we need to load the generated tasks. It might worth running verification again in case we are not sure if it was done before.

In [45]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"

domain = load_domain(DOMAIN_PATH)

In [46]:
# Re-verify all tasks in the saved file (same as `synbench verify`)
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


task_path = OUT_DIR / "tasks.json"
with open(task_path) as f:
    saved = json.load(f)


for item in saved["tasks"]:
    task = Task.model_validate(item)
    # Verify a single task
    verification_result = verify_draft(domain, task)
    # Verification report includes:
    # - passed: bool
    # - errors: list[str]
    # - target_db_hash: str | None
    # - warnings: list[str]
    status = "OK" if verification_result.verification_report.passed else "FAIL"
    print(f"  [{status}] {task.id}")

  [OK] gen_739597cc_0
  [OK] gen_dae9dfa7_1
  [OK] gen_d5b1dd1e_2


In [47]:
# Inspect one task verification
from aieng.syn_data.synbench.verification.pipeline import verify_draft


# Load the first task as an example
task = Task.model_validate(saved["tasks"][0])

result = verify_draft(domain, task)
report = result.verification_report

print("Passed:", report.passed)
print("Errors:", report.errors)
print(
    "Target DB hash:",
    report.target_db_hash[:32] if report.target_db_hash else None,
    "...",
)
print("\nVerified task ready for eval:", result.task.id)

Passed: True
Errors: []
Target DB hash: 9970b161e0dd6be706f2067662f47aef ...

Verified task ready for eval: gen_739597cc_0
